# Eksperimen Machine Learning - Wine Quality Dataset

**Nama:** Rizky  
**Username Dicoding:** rzky0x  
**Dataset:** Wine Quality (Red Wine) - UCI Machine Learning Repository  
**Tujuan:** Klasifikasi kualitas wine (Good/Bad) berdasarkan fitur physicochemical  

---

## 1. Data Loading

Pada tahap ini, kita akan memuat dataset Wine Quality (Red Wine) dari UCI ML Repository.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('Libraries imported successfully!')

In [ ]:
# Load dataset
# Coba load dari local terlebih dahulu, jika tidak ada download dari UCI
import os

local_path = '../wine_quality_raw/winequality-red.csv'
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'

if os.path.exists(local_path):
    df = pd.read_csv(local_path, sep=';')
    print(f'Data loaded from local file: {local_path}')
else:
    df = pd.read_csv(url, sep=';')
    print(f'Data downloaded from: {url}')
    # Save to local
    os.makedirs('../wine_quality_raw', exist_ok=True)
    df.to_csv(local_path, sep=';', index=False)
    print(f'Data saved to: {local_path}')

print(f'\nDataset shape: {df.shape}')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')

In [ ]:
# Preview data
print('=== First 5 Rows ===')
df.head()

In [ ]:
# Dataset info
print('=== Dataset Info ===')
df.info()

In [ ]:
# Column names
print('=== Column Names ===')
for i, col in enumerate(df.columns, 1):
    print(f'{i}. {col} (dtype: {df[col].dtype})')

## 2. Exploratory Data Analysis (EDA)

Pada tahap ini, kita akan melakukan eksplorasi data untuk memahami distribusi, korelasi, dan pola dalam dataset.

In [ ]:
# Descriptive statistics
print('=== Descriptive Statistics ===')
df.describe()

In [ ]:
# Missing values check
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

In [ ]:
# Duplicate rows
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates}')
print(f'Percentage: {duplicates/len(df)*100:.2f}%')

In [ ]:
# Target variable distribution
print('=== Quality Distribution ===')
quality_dist = df['quality'].value_counts().sort_index()
print(quality_dist)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
axes[0].bar(quality_dist.index, quality_dist.values, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Wine Quality Scores')
for i, v in enumerate(quality_dist.values):
    axes[0].text(quality_dist.index[i], v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(quality_dist.values, labels=quality_dist.index, autopct='%1.1f%%', 
            colors=plt.cm.Set3.colors[:len(quality_dist)])
axes[1].set_title('Quality Score Proportion')

plt.tight_layout()
plt.savefig('quality_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature distributions
feature_cols = [col for col in df.columns if col != 'quality']

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=12)
    axes[i].set_xlabel('')
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'mean={df[col].mean():.2f}')
    axes[i].legend(fontsize=8)

# Hide unused subplot
axes[-1].set_visible(False)

plt.suptitle('Feature Distributions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14, 10))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, square=True)
plt.title('Correlation Heatmap - Wine Quality Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlation with target
print('\n=== Correlation with Quality ===')
corr_with_quality = corr_matrix['quality'].drop('quality').sort_values(ascending=False)
print(corr_with_quality)

In [ ]:
# Box plots by quality
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    df.boxplot(column=col, by='quality', ax=axes[i])
    axes[i].set_title(col, fontsize=12)
    axes[i].set_xlabel('Quality')

axes[-1].set_visible(False)

plt.suptitle('Feature Distribution by Quality Score', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('boxplots_by_quality.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Outlier detection using IQR
print('=== Outlier Detection (IQR Method) ===')
for col in feature_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f'{col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%)')

## 3. Data Preprocessing

Pada tahap ini, kita akan melakukan preprocessing data:
1. Handle missing values
2. Remove duplicates
3. Create binary target (quality > 5 = Good, else Bad)
4. Feature scaling (StandardScaler)
5. Train/test split (stratified)

In [ ]:
# Step 1: Handle missing values
print('=== Handle Missing Values ===')
df_processed = df.copy()

missing_before = df_processed.isnull().sum().sum()
print(f'Missing values before: {missing_before}')

# Fill numeric columns with median (if any missing)
numeric_cols = df_processed.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_processed[col].isnull().sum() > 0:
        median_val = df_processed[col].median()
        df_processed[col].fillna(median_val, inplace=True)
        print(f'  Filled {col} with median: {median_val}')

missing_after = df_processed.isnull().sum().sum()
print(f'Missing values after: {missing_after}')

In [ ]:
# Step 2: Remove duplicates
print('=== Remove Duplicates ===')
print(f'Shape before: {df_processed.shape}')
duplicates = df_processed.duplicated().sum()
df_processed = df_processed.drop_duplicates()
print(f'Duplicates removed: {duplicates}')
print(f'Shape after: {df_processed.shape}')

In [ ]:
# Step 3: Create binary target
print('=== Create Binary Target ===')
print(f'Original quality range: {df_processed["quality"].min()} - {df_processed["quality"].max()}')

df_processed['quality_label'] = (df_processed['quality'] > 5).astype(int)

print(f'\nBinary distribution:')
print(f'  Bad  (quality <= 5, label=0): {(df_processed["quality_label"] == 0).sum()}')
print(f'  Good (quality >  5, label=1): {(df_processed["quality_label"] == 1).sum()}')

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
df_processed['quality_label'].value_counts().plot(kind='bar', color=['#e74c3c', '#2ecc71'], ax=ax)
ax.set_xticklabels(['Bad (0)', 'Good (1)'], rotation=0)
ax.set_title('Binary Quality Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('binary_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Step 4: Separate features and target
feature_cols = [col for col in df_processed.columns if col not in ['quality', 'quality_label']]
X = df_processed[feature_cols].copy()
y = df_processed['quality_label'].copy()

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nFeature columns: {feature_cols}')

In [ ]:
# Step 5: Train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'=== Train/Test Split ===')
print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'y_train: {y_train.shape} (Bad: {(y_train==0).sum()}, Good: {(y_train==1).sum()})')
print(f'y_test:  {y_test.shape} (Bad: {(y_test==0).sum()}, Good: {(y_test==1).sum()})')

In [ ]:
# Step 6: Feature scaling
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=feature_cols,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=feature_cols,
    index=X_test.index
)

print('=== Feature Scaling (StandardScaler) ===')
print('Scaled train data statistics:')
print(X_train_scaled.describe().round(2))

In [ ]:
# Step 7: Save preprocessed data
import os

output_dir = 'wine_quality_preprocessing'
os.makedirs(output_dir, exist_ok=True)

# Combine features and target
train_df = X_train_scaled.copy()
train_df['quality_label'] = y_train

test_df = X_test_scaled.copy()
test_df['quality_label'] = y_test

# Save
train_df.to_csv(os.path.join(output_dir, 'train.csv'), index=False)
test_df.to_csv(os.path.join(output_dir, 'test.csv'), index=False)

# Full preprocessed dataset
full_df = pd.concat([train_df, test_df], axis=0)
full_df.to_csv(os.path.join(output_dir, 'wine_quality_preprocessed.csv'), index=False)

print(f'Preprocessed data saved to: {output_dir}/')
print(f'  train.csv: {train_df.shape[0]} rows')
print(f'  test.csv: {test_df.shape[0]} rows')
print(f'  wine_quality_preprocessed.csv: {full_df.shape[0]} rows')

## Summary

### Dataset
- **Source:** UCI ML Repository - Wine Quality (Red Wine)
- **Total samples:** 1,599
- **Features:** 11 physicochemical properties
- **Target:** Binary classification (Good: quality > 5, Bad: quality <= 5)

### Preprocessing Steps
1. ✅ Handle missing values (median imputation)
2. ✅ Remove duplicate rows
3. ✅ Create binary target variable
4. ✅ Train/test split (80/20, stratified)
5. ✅ Feature scaling (StandardScaler)
6. ✅ Save preprocessed data

### Key Findings from EDA
- No missing values in the original dataset
- Some duplicate rows found and removed
- Class imbalance exists (more bad wine than good)
- Alcohol has the highest positive correlation with quality
- Volatile acidity has the highest negative correlation with quality